# Appendix FID — diagnostic depths + split-half noise floor

Fills two pending appendix tables from the paper draft
(`tab:fid_full_appendix` and `tab:fid_splithalf`).

Two outputs:

1. **tab:fid_full_appendix** — FID for the 6 generative methods at the
   $\phi$-decoder and flux-head depths (the diagnostic depths kept out of
   the main table).
2. **tab:fid_splithalf** — split-half noise validation on a representative
   held-out operating point: 2K samples are drawn, partitioned into
   disjoint halves $\mathcal{H}_A,\mathcal{H}_B$, and FID is reported for
   each half against the same reference set, at all four depths
   (skip $L{=}1$, skip $L{=}2$, $\phi$-decoder, flux head).

In [1]:
%load_ext autoreload
%autoreload 2

import os, sys
os.environ["CUDA_VISIBLE_DEVICES"] = "3"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

from pathlib import Path
PROJECT_ROOT = "/system/user/gutenbru/pl/plasmamodelling"
for _p in (PROJECT_ROOT, os.path.join(PROJECT_ROOT, "notebooks")):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import pandas as pd
import torch

from notebooks.neurips_fid_gyroswin_latents import (
    LatentSource, setup, collect_real,
    split_by_traj_set, extract_features, compute_fid_set,
)
from notebooks.neurips_table2_fid import (
    sample_vae, sample_vqvae, sample_ar,
    sample_diff_inplace, sample_diff, split_half_fid,
)
from notebooks.neurips_generate_table1 import (
    TRAJECTORIES_ID, TRAJECTORIES_OOD, TRAJECTORIES_TEST,
    free_cuda, _traj_basename,
    evaluate_reconstruction, build_recon_table, format_recon_latex,
)

torch.use_deterministic_algorithms(False)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"DEVICE = {DEVICE}")

DEVICE = cuda


In [2]:
# ---- paths ---------------------------------------------------------
CKPT_ROOT           = "/restricteddata/ukaea/checkpoints/neurips26"
VAE_CKPT_DIR        = f"{CKPT_ROOT}/VAE_noCond/20260412_174027_099"
VQVAE_CKPT_DIR      = f"{CKPT_ROOT}/VQVAE_decCond/20260409_111450_134"
AR_CKPT_DIR         = f"{CKPT_ROOT}/VQVAE_AR/20260416_130636_591"
DIFF_FLOW_CKPT_DIR  = f"{CKPT_ROOT}/DIFF_FLOW/20260412_180101_948"
DIFF_EDM_CKPT_DIR   = f"{CKPT_ROOT}/DIFF_EDM/20260412_180056_968"
DIFF_DDPM_CKPT_DIR  = f"{CKPT_ROOT}/DIFF_DDPM/20260412_180102_305"
AE_CHECKPOINT       = f"{CKPT_ROOT}/AE_noCond/20260405_022851_327/best.pth"
VQ_INDEX_PKL        = f"{CKPT_ROOT}/norm_files/diff_train_indices_offset80_mu_57f5caf20c1e_indices_vqvae134.pkl"
DATA_PREP           = Path("/restricteddata/ukaea/gyrokinetics/preprocessed_kvikio")
INFERENCE_CFG       = f"{PROJECT_ROOT}/configs/pinc_inference.yaml"

# ---- pick ONE GyroSwin variant ('old' or 'new') --------------------
GYROSWIN_USE = "old"
GYROSWIN_OLD = "/restricteddata/ukaea/checkpoints/scaling_law/gyroswin_xxl_fluxavg_cond_nodrop_l1"
GYROSWIN_NEW = f"{CKPT_ROOT}/GyroSwin_warm/20260422_081638_223"

GYROSWIN_CHECKPOINT     = GYROSWIN_OLD if GYROSWIN_USE == "old" else None
GYROSWIN_NEW_CHECKPOINT = GYROSWIN_NEW if GYROSWIN_USE == "new" else None
print(f"GyroSwin variant in use: {GYROSWIN_USE}")

GyroSwin variant in use: old


## Knobs

* **`K_APPENDIX = 64`** — number of generated samples per traj per method
  for the appendix table (matches the paper's main-text $K$).
* **`K_SPLITHALF = 64`** — half size for the split-half check (so 2K=128
  samples are drawn for the chosen method on the chosen trajectory).
* **`N_REF_PER_TRAJ`** — number of real validation snapshots per traj
  used as the reference set ($N_{\mathrm{ref}}\!\sim\!80$ in the paper).
* **`SPLITHALF_METHOD`** — which generative method gets the noise check
  (paper notes "a representative held-out operating point").
* **`SPLITHALF_TRAJ`** — basename of the validation trajectory used for
  split-half (must be in `TRAJECTORIES_ID + TRAJECTORIES_OOD`).

In [3]:
# ---- knobs ----------------------------------------------------------
N_REF_PER_TRAJ      = 80
K_APPENDIX          = 64
K_SPLITHALF         = 64
GEN_BATCH_SIZE      = 32
AR_BATCH_SIZE       = 8
FEAT_BATCH_SIZE     = 8
N_DENOISING_STEPS   = 15
PCA_COMPONENTS      = 64
SEED                = 42

# ---- which method + trajectory drives the split-half noise floor ----
# Defaults: use the headline paper method (Flow Matching) on a held-out
# ID trajectory.
SPLITHALF_METHOD = "Diff FLOW"
SPLITHALF_TRAJ   = "iteration_148"   # basename, must match a valset file

# ---- the four depths the appendix tracks ----------------------------
# `extract_gyroswin_latents` source/level conventions:
#   skip   level=-2 -> second-deepest down-block skip ("L=1" above bottleneck)
#   skip   level=-1 -> deepest down-block skip       ("L=2", bottleneck-adjacent)
#   phi               -> phi-side bottleneck activation
#   flux_head         -> concat of all multiscale flux_head levels (pre-MLP)
# DEPTHS = [
#     LatentSource("skip L=1",     source="skip",      level=-2),
#     LatentSource("skip L=2",     source="skip",      level=-1),
#     LatentSource("phi decoder",  source="phi"),
#     LatentSource("flux head",    source="flux_head"),
# ]
DEPTHS = [
    LatentSource("skip L=1 (bottleneck)", source="bottleneck", pool="amax"),
    LatentSource("skip L=2",              source="skip",       level=-1, pool="amax"),
    LatentSource("phi decoder",           source="phi",        pool="amax"),
    LatentSource("flux head",             source="flux_head",  level=1),
]
DEPTHS_APPENDIX = [d for d in DEPTHS if d.name in ("phi decoder", "flux head")]

# ---- methods compared in the appendix table ------------------------
RUN_VAE       = True
RUN_VQVAE     = True
RUN_AR        = True
RUN_DIFF_FLOW = True
RUN_DIFF_EDM  = True
RUN_DIFF_DDPM = True

# All splits feed the runner's validation dataset so the FID table can
# pool ID + OOD + TEST trajectories under one valset.
VALID_H5 = [
    *(t.replace("_ifft_realpotens", "") + ".h5" for t in TRAJECTORIES_ID),
    *(t.replace("_ifft_realpotens", "") + ".h5" for t in TRAJECTORIES_OOD),
    *(t.replace("_ifft_realpotens", "") + ".h5" for t in TRAJECTORIES_TEST),
]
print(f"trajectories: ID={len(TRAJECTORIES_ID)}  OOD={len(TRAJECTORIES_OOD)}  "
      f"TEST={len(TRAJECTORIES_TEST)}  |  depths={len(DEPTHS)}  "
      f"|  split-half: {SPLITHALF_METHOD} on {SPLITHALF_TRAJ}")

trajectories: ID=6  OOD=5  TEST=3  |  depths=4  |  split-half: Diff FLOW on iteration_148


## Build runner + chosen GyroSwin

In [4]:
runner, gyroswins = setup(
    DIFF_FLOW_CKPT_DIR, AE_CHECKPOINT, DATA_PREP,
    valid_traj_h5_names=VALID_H5, device=DEVICE,
    gyroswin_checkpoint=GYROSWIN_CHECKPOINT,
    gyroswin_checkpoint_new=GYROSWIN_NEW_CHECKPOINT,
)
assert len(gyroswins) == 1, "this notebook expects exactly one GyroSwin variant"
GS = next(iter(gyroswins.values()))     # single-variant info dict
GS_VARIANT = next(iter(gyroswins.keys()))
print(f"GyroSwin [{GS_VARIANT}]: cond_keys={GS['cond_keys']}  "
      f"# up_blocks={len(GS['model'].df_unet.up_blocks)}  "
      f"# down_blocks={len(GS['model'].df_unet.down_blocks)}")

# Resolve the split-half trajectory to a valset file index. `_traj_basename`
# strips both `.h5` and the `_ifft_realpotens` suffix that the valset uses.
valset = runner.valsets[0]
split_fi = next(
    (fi for fi, f in enumerate(valset.files)
     if _traj_basename(f) == SPLITHALF_TRAJ),
    None,
)
assert split_fi is not None, (
    f"SPLITHALF_TRAJ={SPLITHALF_TRAJ!r} not found among valset files "
    f"({[_traj_basename(f) for f in valset.files]})"
)
print(f"split-half traj resolved: {SPLITHALF_TRAJ} -> fi={split_fi}")

Diffusion latent dataset mode: AE
Loaded AE config for normalization from /restricteddata/ukaea/checkpoints/neurips26/AE_noCond/20260405_022851_327/config.yaml
Loading ['df', 'phi', 'flux'] in dataset
    metadata: 289 files in 5.4s (lightweight=True)
loading aggregated stats from /restricteddata/ukaea/gyrokinetics/preprocessed_kvikio/diff_df_flux_phi_offset80_mu_57f5caf2_df01345_phi012_agg_stats.pkl
    metadata: 14 files in 0.5s (lightweight=True)
Train: 44400
Holdout trajectories (val): 2590
Validation ratio: 0.06
[init] RSS after setup_data: 1.3 GB


/system/apps/userenv/galletti/mhd/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -laio: No such file or directory
collect2: error: ld returned 1 exit status
/system/apps/userenv/galletti/mhd/bin/../lib/gcc/x86_64-conda-linux-gnu/11.2.0/../../../../x86_64-conda-linux-gnu/bin/ld: cannot find -lcufile: No such file or directory
collect2: error: ld returned 1 exit status


AE parameters: 217.6M
Compression: 604.4x (type: ae). (32, *[32, 16, 85, 32]) -> (256, *[4, 2, 9, 4])
Loading model /restricteddata/ukaea/checkpoints/neurips26/AE_noCond/20260405_022851_327/best.pth (stopped at epoch 400) 
loading precomputed latents from /restricteddata/ukaea/gyrokinetics/preprocessed_kvikio/diff_train_latents_offset80_mu_57f5caf20c1e_latents_ae327.pkl
latent_scaling_mode: global
latent stats shape: (1, 1, 1, 1, 1)
latent mean: 0.6872391104698181
latent var: 437.4828796386719
latent l2 norm: 5643.681640625
latent stats -> mean(var): 4.374829e+02, latent_scale [global]: 0.047810
Parameters: 120.5M
[init] RSS after setup_components: 35.5 GB
  diff ckpt epoch=120, params=120.5M


/restricteddata/ukaea/checkpoints/scaling_law/gyroswin_xxl_fluxavg_cond_nodrop_l1/src/models/nd_vit/positional.py:35: UserWarning: Sincos initialization only works if len(grid_size) < dim(returns zero padding otherwise). Switching to random
  warn(


Parameters: 995.1M
GyroSwin [old]: cond_keys=['dg', 'itg', 'q', 's_hat', 'timestep']  # up_blocks=1  # down_blocks=1
split-half traj resolved: iteration_148 -> fi=10


## Collect real validation snapshots

`physical=True` denormalizes via the diffusion runner's valset stats so
the per-batch GS-renormalization in `extract_features` puts everything
into the GyroSwin checkpoint's training-time space.

In [5]:
# --- in-memory metadata patch ---------------------------------------
# Some upstream trajectory metadata.pkl files store the per-step flux
# array under the legacy key `fluxes` (plural) instead of `flux`
# (singular) which the dataset loader expects. Rename in-memory so
# `_load_data` can read `meta["flux"][...]`. The disk pkls are read-only
# so we can't fix them at the source; this is a no-op for files that
# already have `flux`.
def _patch_legacy_flux(dataset):
    patched = []
    for fi, meta in dataset.metadata.items():
        if "flux" not in meta and "fluxes" in meta:
            meta["flux"] = meta["fluxes"]
            patched.append(fi)
    if patched:
        print(f"  [patch] renamed `fluxes`->`flux` on metadata fi in {patched}")
    return patched

_patch_legacy_flux(valset)
_patch_legacy_flux(runner.trainset)   # keep trainset consistent for sample_nn

# Sanity check: every valset entry now has `flux`.
_bad = [(fi, p) for fi, p in enumerate(valset.files)
        if "flux" not in valset.metadata.get(fi, {})]
assert not _bad, f"valset entries still missing 'flux' after patch: {_bad}"

real_by_fi = collect_real(runner, n_per_traj=N_REF_PER_TRAJ, physical=True)
real_split = split_by_traj_set(
    real_by_fi, runner,
    TRAJECTORIES_ID, TRAJECTORIES_OOD, TRAJECTORIES_TEST,
)
assert split_fi in real_by_fi, (
    f"fi={split_fi} ({SPLITHALF_TRAJ}) missing from real_by_fi -- "
    "adjust SPLITHALF_TRAJ."
)
real_split_half = {split_fi: real_by_fi[split_fi]}

  [patch] renamed `fluxes`->`flux` on metadata fi in [9]
  real samples: 1120 across 14 trajs ({0: 80, 1: 80, 2: 80, 3: 80, 4: 80, 5: 80, 6: 80, 7: 80, 8: 80, 9: 80, 10: 80, 11: 80, 12: 80, 13: 80})  [physical]
  [split] ID: 6 trajs | OOD: 5 trajs | TEST: 3 trajs


## Generate $K$-sample sets for the appendix table

One pass per method, $K=64$ samples per traj. Models are torn down
between methods so we never hold more than one at a time on the GPU.

In [6]:
gen_by_source = {}

if RUN_VAE:
    print("\n========================  VAE  ========================")
    gen_by_source["VAE"] = sample_vae(
        VAE_CKPT_DIR, runner, real_by_fi,
        n_per_traj=K_APPENDIX, batch_size=GEN_BATCH_SIZE,
        data_path=DATA_PREP, device=DEVICE,
    ); free_cuda()

if RUN_VQVAE:
    print("\n========================  VQ-VAE + random  ========================")
    gen_by_source["VQ-VAE + random"] = sample_vqvae(
        VQVAE_CKPT_DIR, VQ_INDEX_PKL, runner, real_by_fi,
        n_per_traj=K_APPENDIX, batch_size=GEN_BATCH_SIZE,
        data_path=DATA_PREP, device=DEVICE,
    ); free_cuda()

if RUN_AR:
    print("\n========================  VQ-VAE + Transformer  ========================")
    gen_by_source["VQ-VAE + Transformer"] = sample_ar(
        AR_CKPT_DIR, VQVAE_CKPT_DIR, runner, real_by_fi,
        n_per_traj=K_APPENDIX, batch_size=AR_BATCH_SIZE,
        data_path=DATA_PREP, device=DEVICE,
    ); free_cuda()

if RUN_DIFF_FLOW:
    print("\n========================  Diff FLOW  ========================")
    gen_by_source["Diff FLOW"] = sample_diff_inplace(
        runner, real_by_fi,
        n_per_traj=K_APPENDIX,
        n_denoising_steps=N_DENOISING_STEPS, batch_size=GEN_BATCH_SIZE,
    ); free_cuda()

if RUN_DIFF_EDM:
    print("\n========================  Diff EDM  ========================")
    gen_by_source["Diff EDM"] = sample_diff(
        DIFF_EDM_CKPT_DIR, AE_CHECKPOINT, runner, real_by_fi,
        n_per_traj=K_APPENDIX,
        n_denoising_steps=N_DENOISING_STEPS, batch_size=GEN_BATCH_SIZE,
        data_path=DATA_PREP, device=DEVICE,
    ); free_cuda()

if RUN_DIFF_DDPM:
    print("\n========================  Diff DDPM  ========================")
    gen_by_source["Diff DDPM"] = sample_diff(
        DIFF_DDPM_CKPT_DIR, AE_CHECKPOINT, runner, real_by_fi,
        n_per_traj=K_APPENDIX,
        n_denoising_steps=N_DENOISING_STEPS, batch_size=GEN_BATCH_SIZE,
        data_path=DATA_PREP, device=DEVICE,
    ); free_cuda()

gen_split_by_source = {
    name: split_by_traj_set(
        d, runner,
        TRAJECTORIES_ID, TRAJECTORIES_OOD, TRAJECTORIES_TEST,
    )
    for name, d in gen_by_source.items()
}
print("\nappendix sources ready:", list(gen_by_source.keys()))


========================  VAE  ========================
  loading AE trainset stats (/restricteddata/ukaea/checkpoints/neurips26/VAE_noCond/20260412_174027_099) ...
Loading ['df', 'phi', 'flux'] in dataset
    metadata: 289 files in 6.8s (lightweight=True)
loading pre-computed stats from /restricteddata/ukaea/gyrokinetics/preprocessed_kvikio/diff_df_flux_phi_offset80_mu_57f5caf2_stats.pkl
No aggregations for flux, using stats of shape: (1,)
    metadata: 6 files in 0.1s (lightweight=True)
Train: 44400
Holdout trajectories (val): 1110
Validation ratio: 0.03
  loading AE weights (/restricteddata/ukaea/checkpoints/neurips26/VAE_noCond/20260412_174027_099) ...
VAE parameters: 217.9M
Compression: 604.4x (type: vae). (32, *[32, 16, 85, 32]) -> (256, *[4, 2, 9, 4])
Loading model /restricteddata/ukaea/checkpoints/neurips26/VAE_noCond/20260412_174027_099/best.pth (stopped at epoch 400) 


  vae sample: 100%|██████████| 14/14 [01:14<00:00,  5.29s/it]



========================  VQ-VAE + random  ========================
  loading AE trainset stats (/restricteddata/ukaea/checkpoints/neurips26/VQVAE_decCond/20260409_111450_134) ...
Loading ['df', 'phi', 'flux'] in dataset
    metadata: 289 files in 4.1s (lightweight=True)
loading pre-computed stats from /restricteddata/ukaea/gyrokinetics/preprocessed_kvikio/diff_df_flux_phi_offset80_mu_57f5caf2_stats.pkl
No aggregations for flux, using stats of shape: (1,)
    metadata: 3 files in 0.1s (lightweight=True)
Train: 44400
Holdout trajectories (val): 555
Validation ratio: 0.01
  loading AE weights (/restricteddata/ukaea/checkpoints/neurips26/VQVAE_decCond/20260409_111450_134) ...
VQ-VAE parameters: 225.8M
Compression: 95223.2x (type: vqvae). (32, *[32, 16, 85, 32]) -> (256, *[4, 2, 9, 4])
Loading model /restricteddata/ukaea/checkpoints/neurips26/VQVAE_decCond/20260409_111450_134/best.pth (stopped at epoch 400) 


  vqvae sample: 100%|██████████| 14/14 [01:17<00:00,  5.55s/it]



========================  VQ-VAE + Transformer  ========================
  building trainset (AR config: ae_checkpoint=/restricteddata/ukaea/checkpoints/neurips26/VQVAE_decCond/20260409_111450_134) ...
Diffusion latent dataset mode: VQVAE
Loaded AE config for normalization from /restricteddata/ukaea/checkpoints/neurips26/VQVAE_decCond/20260409_111450_134/config.yaml
Loading ['df', 'phi', 'flux'] in dataset
    metadata: 289 files in 4.1s (lightweight=True)
loading aggregated stats from /restricteddata/ukaea/gyrokinetics/preprocessed_kvikio/diff_df_flux_phi_offset80_mu_57f5caf2_df01345_phi012_agg_stats.pkl
    metadata: 6 files in 0.1s (lightweight=True)
Train: 44400
Holdout trajectories (val): 108
Validation ratio: 0.00
  loading AE (VQ-VAE) ...
VQ-VAE parameters: 225.8M
Compression: 95223.2x (type: vqvae). (32, *[32, 16, 85, 32]) -> (256, *[4, 2, 9, 4])
Loading model /restricteddata/ukaea/checkpoints/neurips26/VQVAE_decCond/20260409_111450_134/best.pth (stopped at epoch 400) 
  build

...


========================  Diff EDM  ========================
Diffusion latent dataset mode: AE
Loaded AE config for normalization from /restricteddata/ukaea/checkpoints/neurips26/AE_noCond/20260405_022851_327/config.yaml
Loading ['df', 'phi', 'flux'] in dataset
    metadata: 289 files in 7.0s (lightweight=True)
loading aggregated stats from /restricteddata/ukaea/gyrokinetics/preprocessed_kvikio/diff_df_flux_phi_offset80_mu_57f5caf2_df01345_phi012_agg_stats.pkl
    metadata: 14 files in 0.3s (lightweight=True)
Train: 44400
Holdout trajectories (val): 2590
Validation ratio: 0.06
[init] RSS after setup_data: 824.5 GB
AE parameters: 217.6M
Compression: 604.4x (type: ae). (32, *[32, 16, 85, 32]) -> (256, *[4, 2, 9, 4])
Loading model /restricteddata/ukaea/checkpoints/neurips26/AE_noCond/20260405_022851_327/best.pth (stopped at epoch 400) 
loading precomputed latents from /restricteddata/ukaea/gyrokinetics/preprocessed_kvikio/diff_train_latents_offset80_mu_57f5caf20c1e_latents_ae327.pkl
late

  20260412_180056_968 sample: 100%|██████████| 14/14 [01:23<00:00,  5.94s/it]



========================  Diff DDPM  ========================
Diffusion latent dataset mode: AE
Loaded AE config for normalization from /restricteddata/ukaea/checkpoints/neurips26/AE_noCond/20260405_022851_327/config.yaml
Loading ['df', 'phi', 'flux'] in dataset
    metadata: 289 files in 4.5s (lightweight=True)
loading aggregated stats from /restricteddata/ukaea/gyrokinetics/preprocessed_kvikio/diff_df_flux_phi_offset80_mu_57f5caf2_df01345_phi012_agg_stats.pkl
    metadata: 14 files in 0.4s (lightweight=True)
Train: 44400
Holdout trajectories (val): 2590
Validation ratio: 0.06
[init] RSS after setup_data: 999.8 GB
AE parameters: 217.6M
Compression: 604.4x (type: ae). (32, *[32, 16, 85, 32]) -> (256, *[4, 2, 9, 4])
Loading model /restricteddata/ukaea/checkpoints/neurips26/AE_noCond/20260405_022851_327/best.pth (stopped at epoch 400) 
loading precomputed latents from /restricteddata/ukaea/gyrokinetics/preprocessed_kvikio/diff_train_latents_offset80_mu_57f5caf20c1e_latents_ae327.pkl
lat

  20260412_180102_305 sample: 100%|██████████| 14/14 [01:28<00:00,  6.30s/it]


  [split] ID: 6 trajs | OOD: 5 trajs | TEST: 3 trajs
  [split] ID: 6 trajs | OOD: 5 trajs | TEST: 3 trajs
  [split] ID: 6 trajs | OOD: 5 trajs | TEST: 3 trajs
  [split] ID: 6 trajs | OOD: 5 trajs | TEST: 3 trajs
  [split] ID: 6 trajs | OOD: 5 trajs | TEST: 3 trajs
  [split] ID: 6 trajs | OOD: 5 trajs | TEST: 3 trajs

appendix sources ready: ['VAE', 'VQ-VAE + random', 'VQ-VAE + Transformer', 'Diff FLOW', 'Diff EDM', 'Diff DDPM']


## tab:fid_full_appendix — global FID at $\phi$-decoder + flux-head depths

Pooled real-vs-gen FID across the **ID + OOD** validation set, exactly
as in `tab:fid` but at the two diagnostic depths.

In [7]:
# Pool ID + OOD + TEST into a single by_fi dict for each source. The
# FID table is reported globally across all evaluated
# trajectories; including TEST keeps the table compatible while still
# letting per-split breakdowns be derived from `real_split` /
# `gen_split_by_source` if needed.
def _pool_splits(d):
    return {fi: e for split in ("ID", "OOD", "TEST")
            for fi, e in d.get(split, {}).items()}

real_pooled = _pool_splits(real_split)
gen_pooled = {src: _pool_splits(g) for src, g in gen_split_by_source.items()}

appendix_rows = []
for ls in DEPTHS_APPENDIX:
    print(f"\n--- depth: {ls.name} ---")
    real_feats = extract_features(
        GS["model"], real_pooled, runner, GS["cond_keys"], ls,
        batch_size=FEAT_BATCH_SIZE, device=DEVICE,
        desc=f"real:{ls.name}",
        gs_norm_stats=GS["norm_stats"],
    )
    for source_name, gen_dict in gen_pooled.items():
        gen_feats = extract_features(
            GS["model"], gen_dict, runner, GS["cond_keys"], ls,
            batch_size=FEAT_BATCH_SIZE, device=DEVICE,
            desc=f"{source_name}:{ls.name}",
            gs_norm_stats=GS["norm_stats"],
        )
        res = compute_fid_set(real_feats, gen_feats, n_components=PCA_COMPONENTS)
        appendix_rows.append({
            "depth":  ls.name,
            "method": source_name,
            "FID":    res["fid_global"],
        })
        print(f"  {source_name:<22}  FID = {res['fid_global']:.4f}")
        free_cuda()

appendix_df = (
    pd.DataFrame(appendix_rows)
      .pivot(index="method", columns="depth", values="FID")
      .reindex(columns=[d.name for d in DEPTHS_APPENDIX])
)
# Keep the row ordering from the paper draft.
_paper_method_order = [
    "VAE", "VQ-VAE + random", "VQ-VAE + Transformer",
    "Diff DDPM", "Diff FLOW", "Diff EDM",
]
appendix_df = appendix_df.reindex(
    [m for m in _paper_method_order if m in appendix_df.index]
)
print("\ntab:fid_full_appendix")
print(appendix_df.to_string(float_format=lambda x: f"{x:.4f}"))
appendix_df


--- depth: phi decoder ---
The history saving thread hit an unexpected error (OperationalError('unable to open database file')).History will not be written to the database.
  VAE                     FID = 880481.2051
  VQ-VAE + random         FID = 109918.6919
  VQ-VAE + Transformer    FID = 70766.7200
  Diff FLOW               FID = 246953.3979
  Diff EDM                FID = 201317.1898
  Diff DDPM               FID = 685426.2303

--- depth: flux head ---
  VAE                     FID = 383.0090
  VQ-VAE + random         FID = 95.1756
  VQ-VAE + Transformer    FID = 50.3370
  Diff FLOW               FID = 38.6148
  Diff EDM                FID = 38.4753
  Diff DDPM               FID = 480.0413

tab:fid_full_appendix
depth                 phi decoder  flux head
method                                      
VAE                   880481.2051   383.0090
VQ-VAE + random       109918.6919    95.1756
VQ-VAE + Transformer   70766.7200    50.3370
Diff DDPM             685426.2303   480.0413
Di

depth,phi decoder,flux head
method,,
VAE,880481.205068,383.009036
VQ-VAE + random,109918.691936,95.175563
VQ-VAE + Transformer,70766.719961,50.337021
Diff DDPM,685426.230326,480.041264
Diff FLOW,246953.397930,38.614776
Diff EDM,201317.189842,38.475255


## Sample $2K$ generations for the split-half check

Single trajectory (`SPLITHALF_TRAJ`), single method (`SPLITHALF_METHOD`):
we draw 2K=128 samples in a fresh pass so they're independent of the
appendix samples above.

In [8]:
n_2k = 2 * K_SPLITHALF
if SPLITHALF_METHOD == "VAE":
    sh_dict = sample_vae(
        VAE_CKPT_DIR, runner, real_split_half,
        n_per_traj=n_2k, batch_size=GEN_BATCH_SIZE,
        data_path=DATA_PREP, device=DEVICE,
    )
elif SPLITHALF_METHOD == "VQ-VAE + random":
    sh_dict = sample_vqvae(
        VQVAE_CKPT_DIR, VQ_INDEX_PKL, runner, real_split_half,
        n_per_traj=n_2k, batch_size=GEN_BATCH_SIZE,
        data_path=DATA_PREP, device=DEVICE,
    )
elif SPLITHALF_METHOD == "VQ-VAE + Transformer":
    sh_dict = sample_ar(
        AR_CKPT_DIR, VQVAE_CKPT_DIR, runner, real_split_half,
        n_per_traj=n_2k, batch_size=AR_BATCH_SIZE,
        data_path=DATA_PREP, device=DEVICE,
    )
elif SPLITHALF_METHOD == "Diff FLOW":
    sh_dict = sample_diff_inplace(
        runner, real_split_half,
        n_per_traj=n_2k,
        n_denoising_steps=N_DENOISING_STEPS, batch_size=GEN_BATCH_SIZE,
    )
elif SPLITHALF_METHOD == "Diff EDM":
    sh_dict = sample_diff(
        DIFF_EDM_CKPT_DIR, AE_CHECKPOINT, runner, real_split_half,
        n_per_traj=n_2k,
        n_denoising_steps=N_DENOISING_STEPS, batch_size=GEN_BATCH_SIZE,
        data_path=DATA_PREP, device=DEVICE,
    )
elif SPLITHALF_METHOD == "Diff DDPM":
    sh_dict = sample_diff(
        DIFF_DDPM_CKPT_DIR, AE_CHECKPOINT, runner, real_split_half,
        n_per_traj=n_2k,
        n_denoising_steps=N_DENOISING_STEPS, batch_size=GEN_BATCH_SIZE,
        data_path=DATA_PREP, device=DEVICE,
    )
else:
    raise ValueError(f"unknown SPLITHALF_METHOD={SPLITHALF_METHOD!r}")
free_cuda()
n_drawn = len(sh_dict[split_fi]["df"])
print(f"split-half: drew {n_drawn} samples from {SPLITHALF_METHOD} for {SPLITHALF_TRAJ}")
assert n_drawn >= n_2k

  diff (ref) sample: 100%|██████████| 1/1 [00:11<00:00, 11.42s/it]


split-half: drew 128 samples from Diff FLOW for iteration_148


## tab:fid_splithalf — within-condition noise floor

For each depth, fit one PCA basis on the pooled (real + 2K-gen) features
of the single split-half trajectory, then compute FID against the same
real reference for the full $\mathcal{H}_A\!\cup\!\mathcal{H}_B$ set
and for each half $\mathcal{H}_A$, $\mathcal{H}_B$.

$|\mathrm{FID}_A-\mathrm{FID}_B|/\mathrm{FID}_\mathrm{full}$ is the
relative within-condition noise; values much smaller than the
between-condition spread certify the depth is not noise-limited.

In [9]:
sh_real_dict = real_split_half             # {split_fi: {df: [...], label}}
sh_gen_dict  = sh_dict                     # {split_fi: {df: [2K, ...], label}}

splithalf_rows = []
for ls in DEPTHS:
    print(f"\n--- split-half depth: {ls.name} ---")
    real_feats_dict = extract_features(
        GS["model"], sh_real_dict, runner, GS["cond_keys"], ls,
        batch_size=FEAT_BATCH_SIZE, device=DEVICE,
        desc=f"real:sh/{ls.name}",
        gs_norm_stats=GS["norm_stats"],
    )
    gen_feats_dict = extract_features(
        GS["model"], sh_gen_dict, runner, GS["cond_keys"], ls,
        batch_size=FEAT_BATCH_SIZE, device=DEVICE,
        desc=f"gen:sh/{ls.name}",
        gs_norm_stats=GS["norm_stats"],
    )
    real_feats = real_feats_dict[split_fi]["feats"]
    gen_feats  = gen_feats_dict[split_fi]["feats"]
    res = split_half_fid(
        real_feats, gen_feats,
        k=K_SPLITHALF, n_components=PCA_COMPONENTS, seed=SEED,
    )
    splithalf_rows.append({
        "depth":          ls.name,
        "FID_full":       res["fid_full"],
        "FID_A":          res["fid_a"],
        "FID_B":          res["fid_b"],
        "|A-B|/full":     res["delta_rel"],
        "n_ref":          res["n_ref"],
        "n_gen":          res["n_full"],
        "K":              res["k"],
    })
    print(f"  full={res['fid_full']:.4f}  "
          f"A={res['fid_a']:.4f}  B={res['fid_b']:.4f}  "
          f"|A-B|/full={res['delta_rel']:.3f}")
    free_cuda()

splithalf_df = pd.DataFrame(splithalf_rows).set_index("depth")
splithalf_df = splithalf_df.reindex([d.name for d in DEPTHS])
print(f"\ntab:fid_splithalf  ({SPLITHALF_METHOD} on {SPLITHALF_TRAJ})")
print(splithalf_df.to_string(float_format=lambda x: f"{x:.4f}"))
splithalf_df


--- split-half depth: skip L=1 (bottleneck) ---
  full=6748.5674  A=7377.0636  B=7171.5640  |A-B|/full=0.030

--- split-half depth: skip L=2 ---
  full=11935.2945  A=12216.3454  B=11889.4038  |A-B|/full=0.027

--- split-half depth: phi decoder ---
  full=439264.3304  A=443594.2042  B=438118.2306  |A-B|/full=0.012

--- split-half depth: flux head ---
  full=39.6761  A=44.7353  B=36.8151  |A-B|/full=0.200

tab:fid_splithalf  (Diff FLOW on iteration_148)
                         FID_full       FID_A       FID_B  |A-B|/full  n_ref  n_gen   K
depth                                                                                  
skip L=1 (bottleneck)   6748.5674   7377.0636   7171.5640      0.0305     80    128  64
skip L=2               11935.2945  12216.3454  11889.4038      0.0274     80    128  64
phi decoder           439264.3304 443594.2042 438118.2306      0.0125     80    128  64
flux head                 39.6761     44.7353     36.8151      0.1996     80    128  64


,FID_full,FID_A,FID_B,|A-B|/full,n_ref,n_gen,K
depth,,,,,,,
skip L=1 (bottleneck),6748.567428,7377.063590,7171.563988,0.030451,80,128,64
skip L=2,11935.294512,12216.345429,11889.403834,0.027393,80,128,64
phi decoder,439264.330371,443594.204245,438118.230567,0.012466,80,128,64
flux head,39.676125,44.735292,36.815144,0.199620,80,128,64


### Quick LaTeX dumps (drop straight into the appendix tables)

In [10]:
print("% tab:fid_full_appendix")
print(appendix_df.to_latex(float_format="%.4f", na_rep="--"))
print()
print("% tab:fid_splithalf")
print(
    splithalf_df[["FID_full", "FID_A", "FID_B", "|A-B|/full"]]
    .to_latex(float_format="%.4f", na_rep="--")
)

% tab:fid_full_appendix
\begin{tabular}{lrr}
\toprule
depth & phi decoder & flux head \\
method &  &  \\
\midrule
VAE & 880481.2051 & 383.0090 \\
VQ-VAE + random & 109918.6919 & 95.1756 \\
VQ-VAE + Transformer & 70766.7200 & 50.3370 \\
Diff DDPM & 685426.2303 & 480.0413 \\
Diff FLOW & 246953.3979 & 38.6148 \\
Diff EDM & 201317.1898 & 38.4753 \\
\bottomrule
\end{tabular}


% tab:fid_splithalf
\begin{tabular}{lrrrr}
\toprule
 & FID_full & FID_A & FID_B & |A-B|/full \\
depth &  &  &  &  \\
\midrule
skip L=1 (bottleneck) & 6748.5674 & 7377.0636 & 7171.5640 & 0.0305 \\
skip L=2 & 11935.2945 & 12216.3454 & 11889.4038 & 0.0274 \\
phi decoder & 439264.3304 & 443594.2042 & 438118.2306 & 0.0125 \\
flux head & 39.6761 & 44.7353 & 36.8151 & 0.1996 \\
\bottomrule
\end{tabular}



# Appendix - Reconstruction error (AE / VAE / VQ-VAE)

For each ID + OOD trajectory, runs every post-offset df binary through the
model (encode + decode, eval mode) and computes RMSEs for both:
- `df_RMSE` — point-wise reconstruction RMSE on the distribution function in
  *physical* (denormalized) space.
- `eflux_RMSE`, `kxspec_RMSE`, `kyspec_RMSE`, `fluxspec_RMSE` (= `qspec`) —
  RMSE on the physics integrals (`FluxIntegral(flux_fields=True,
  spectral_df=False, spectral_potens=True)`) computed from the reconstructed
  vs. ground-truth df. Same integrator and column names as the summary table
  above, so the rows are directly comparable.

Per-trajectory RMSE = `sqrt(mean((pred - target)^2))` over all timesteps and
spatial dims for each quantity. The columns below show mean ± sample std
(ddof=1) of those per-traj RMSEs across the trajectories in each split.

`params_M` is the total parameter count of each model in millions.


In [11]:
RECON_BATCH_SIZE = 32

# AE checkpoint here is a path to best.pth -- evaluate_reconstruction takes the
# directory, so strip the trailing filename.
recon_results = {}
recon_results["AE"] = evaluate_reconstruction(
    os.path.dirname(AE_CHECKPOINT), INFERENCE_CFG,
    data_path=DATA_PREP, batch_size=RECON_BATCH_SIZE, device=DEVICE,
    trajectories_test=TRAJECTORIES_TEST,
)
recon_results["VAE"] = evaluate_reconstruction(
    VAE_CKPT_DIR, INFERENCE_CFG,
    data_path=DATA_PREP, batch_size=RECON_BATCH_SIZE, device=DEVICE,
    trajectories_test=TRAJECTORIES_TEST,
)
recon_results["VQ-VAE"] = evaluate_reconstruction(
    VQVAE_CKPT_DIR, INFERENCE_CFG,
    data_path=DATA_PREP, batch_size=RECON_BATCH_SIZE, device=DEVICE,
    trajectories_test=TRAJECTORIES_TEST,
)


  Reconstruction
  ckpt_dir=/restricteddata/ukaea/checkpoints/neurips26/AE_noCond/20260405_022851_327
  GPU before: alloc=5.11G reserved=5.20G
  building trainset (/restricteddata/ukaea/checkpoints/neurips26/AE_noCond/20260405_022851_327) ...
Loading ['df', 'phi', 'flux'] in dataset
    metadata: 289 files in 4.6s (lightweight=True)
loading pre-computed stats from /restricteddata/ukaea/gyrokinetics/preprocessed_kvikio/diff_df_flux_phi_offset80_mu_57f5caf2_stats.pkl
No aggregations for flux, using stats of shape: (1,)
    metadata: 3 files in 0.1s (lightweight=True)
Train: 44400
Holdout trajectories (val): 555
Validation ratio: 0.01
  loading AE weights ...
AE parameters: 217.6M
Compression: 604.4x (type: ae). (32, *[32, 16, 85, 32]) -> (256, *[4, 2, 9, 4])
Loading model /restricteddata/ukaea/checkpoints/neurips26/AE_noCond/20260405_022851_327/best.pth (stopped at epoch 400) 


  iteration_200_ifft_realpotens: 100%|██████████| 6/6 [01:12<00:00, 12.00s/batch]


  GPU after:  alloc=5.11G reserved=5.20G

  Reconstruction
  ckpt_dir=/restricteddata/ukaea/checkpoints/neurips26/VAE_noCond/20260412_174027_099
  GPU before: alloc=5.11G reserved=5.20G
  building trainset (/restricteddata/ukaea/checkpoints/neurips26/VAE_noCond/20260412_174027_099) ...
Loading ['df', 'phi', 'flux'] in dataset
    metadata: 289 files in 4.3s (lightweight=True)
loading pre-computed stats from /restricteddata/ukaea/gyrokinetics/preprocessed_kvikio/diff_df_flux_phi_offset80_mu_57f5caf2_stats.pkl
No aggregations for flux, using stats of shape: (1,)
    metadata: 6 files in 0.2s (lightweight=True)
Train: 44400
Holdout trajectories (val): 1110
Validation ratio: 0.03
  loading AE weights ...
VAE parameters: 217.9M
Compression: 604.4x (type: vae). (32, *[32, 16, 85, 32]) -> (256, *[4, 2, 9, 4])
Loading model /restricteddata/ukaea/checkpoints/neurips26/VAE_noCond/20260412_174027_099/best.pth (stopped at epoch 400) 


  iteration_200_ifft_realpotens: 100%|██████████| 6/6 [01:14<00:00, 12.39s/batch]


  GPU after:  alloc=5.11G reserved=5.20G

  Reconstruction
  ckpt_dir=/restricteddata/ukaea/checkpoints/neurips26/VQVAE_decCond/20260409_111450_134
  GPU before: alloc=5.11G reserved=5.20G
  building trainset (/restricteddata/ukaea/checkpoints/neurips26/VQVAE_decCond/20260409_111450_134) ...
Loading ['df', 'phi', 'flux'] in dataset
    metadata: 289 files in 4.4s (lightweight=True)
loading pre-computed stats from /restricteddata/ukaea/gyrokinetics/preprocessed_kvikio/diff_df_flux_phi_offset80_mu_57f5caf2_stats.pkl
No aggregations for flux, using stats of shape: (1,)
    metadata: 3 files in 0.1s (lightweight=True)
Train: 44400
Holdout trajectories (val): 555
Validation ratio: 0.01
  loading AE weights ...
VQ-VAE parameters: 225.8M
Compression: 95223.2x (type: vqvae). (32, *[32, 16, 85, 32]) -> (256, *[4, 2, 9, 4])
Loading model /restricteddata/ukaea/checkpoints/neurips26/VQVAE_decCond/20260409_111450_134/best.pth (stopped at epoch 400) 


  iteration_200_ifft_realpotens: 100%|██████████| 6/6 [01:15<00:00, 12.52s/batch]


  GPU after:  alloc=5.11G reserved=5.20G


In [12]:
recon_table = build_recon_table(recon_results)
print(recon_table.to_string(float_format=lambda x: f"{x:.4g}"))
recon_table

              params_M  df_RMSE  df_RMSE_std  eflux_RMSE  eflux_RMSE_std  kxspec_RMSE  kxspec_RMSE_std  kyspec_RMSE  kyspec_RMSE_std  fluxspec_RMSE  fluxspec_RMSE_std
model  split                                                                                                                                                          
AE     ID        217.6   0.7533       0.1748       4.765           2.149    1.801e+04        1.881e+04    4.017e+04        4.132e+04         0.4468             0.1327
       OOD       217.6   0.7865       0.2567       7.122           7.346    9.598e+04        1.644e+05    2.149e+05        3.684e+05         0.5181             0.2184
       TEST      217.6   0.6546       0.2509       5.808           2.157         7419        1.067e+04    1.638e+04         2.37e+04         0.4599             0.1182
VAE    ID        217.9   0.8384       0.1707       3.847          0.6758    3.526e+04        4.339e+04    7.574e+04        9.286e+04          1.102             0.395

params_M   df_RMSE  df_RMSE_std  eflux_RMSE  eflux_RMSE_std  \
model  split                                                                  
AE     ID     217.632352  0.753256     0.174764    4.764763        2.149110   
       OOD    217.632352  0.786465     0.256739    7.122474        7.345765   
       TEST   217.632352  0.654600     0.250909    5.808366        2.156535   
VAE    ID     217.894752  0.838366     0.170717    3.846816        0.675843   
       OOD    217.894752  0.868676     0.248892    6.909585        6.370554   
       TEST   217.894752  0.741755     0.244965    6.176152        2.443148   
VQ-VAE ID     225.826528  0.887592     0.187439   21.595732        9.703239   
       OOD    225.826528  0.923898     0.273011   23.496632       13.424471   
       TEST   225.826528  0.779202     0.277379   16.932727        6.664554   

                kxspec_RMSE  kxspec_RMSE_std    kyspec_RMSE  kyspec_RMSE_std  \
model  split                                                                   
AE     ID      18011.359091     18813.887529   40168.456815     41316.521798   
       OOD     95978.187441    164361.566290  214902.475350    368387.416516   
       TEST     7419.120363     10673.262662   16382.516984     23703.940352   
VAE    ID      35255.159024     43388.331360   75737.328451     92861.102985   
       OOD    195096.453946    351422.150713  426025.458196    769335.333615   
       TEST     8772.280678     11413.454136   16976.575863     21545.742898   
VQ-VAE ID      18431.670398     16186.391702   37534.698648     31396.370968   
       OOD     82903.769409    138434.920896  176228.837695    295279.013210   
       TEST     7945.234289      9732.265258   16173.228228     19081.372774   

              fluxspec_RMSE  fluxspec_RMSE_std  
model  split                                    
AE     ID          0.446820           0.132675  
       OOD         0.518143           0.218363  
       TEST        0.459907           0.118177  
VAE    ID          1.102481           0.395171  
       OOD         1.037019           0.409077  
       TEST        0.879596           0.423476  
VQ-VAE ID          1.723139           1.082005  
       OOD         1.418923           0.756762  
       TEST        0.975253           0.450041

In [13]:
# Single-column df-only table (matches the original template)
print(format_recon_latex(recon_results, quantities=["df"]))

# Combined table: df + Q-spec + k_y-spec (3 quantities, 8 columns)
print(format_recon_latex(
    recon_results,
    quantities=["df", "eflux", "kyspec", "qspec"], #["df", "eflux", "kxspec", "kyspec", "qspec"]
    label="tab:ae_recon_combined",
))


\begin{table}[t]
\centering
\caption{Reconstruction RMSE in physical (denormalised) units, evaluated for each autoencoder backbone underlying the latent generative models. Parameter counts in millions (M). ID and OOD entries denote the mean $\pm$ standard deviation across trajectories of the corresponding test split.}
\label{tab:ae_recon_rmse}
\begin{tabular}{lcccc}
\toprule
\multirow{2}{*}{\textbf{Method}} & \multirow{2}{*}{\textbf{Params (M)}} & \multicolumn{3}{c}{${Recon}_{\mathrm{RMSE}}\downarrow$} \\
\cmidrule(lr){3-5}
 &  & \textbf{ID} & \textbf{OOD} & \textbf{TEST} \\
\midrule
AE & 217.6 & $0.753_{\pm 0.175}$ & $0.786_{\pm 0.257}$ & $0.655_{\pm 0.251}$ \\
VAE & 217.9 & $0.838_{\pm 0.171}$ & $0.869_{\pm 0.249}$ & $0.742_{\pm 0.245}$ \\
VQ-VAE & 225.8 & $0.888_{\pm 0.187}$ & $0.924_{\pm 0.273}$ & $0.779_{\pm 0.277}$ \\
\bottomrule
\end{tabular}
\end{table}
\begin{table}[t]
\centering
\caption{Reconstruction RMSE in physical (denormalised) units, evaluated for each autoencoder back